In [1]:
import unicodedata
import re
def clean_text(s):
    # bỏ dấu tiếng Việt
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    
    # giữ lại chữ cái
    s = re.sub(r'[^a-zA-Z]', '', s)
    
    return s.lower()

s = "Tổng sản phẩm trong nước (GDP)!!!"
print(clean_text(s))

tongsanphamtrongnuocgdp


In [ ]:
import pandas as pd

file_path = '../../Bronze_Excel_Financial_Report/2018/IS_Thang/02-Bieu-7.2018-1.xlsx'
excel_file = pd.ExcelFile(file_path)

all_sheets = excel_file.sheet_names

import_index = -1
export_index = -1

for i in range(len(all_sheets)):
    name = clean_text(all_sheets[i])
    if any(sheet in name for sheet in ['nk', 'nhapkhau']) and 'quy' not in name and 'gia' not in name:
        import_index = i
    if any(sheet in name for sheet in ['xk', 'xuatkhau']) and 'quy' not in name and 'gia' not in name:
        export_index = i

import_sheet = pd.read_excel(file_path, sheet_name= all_sheets[import_index], header= None)
export_sheet = pd.read_excel(file_path, sheet_name= all_sheets[export_index], header= None)

def extract_ecommerce_data(sheet : pd.DataFrame, type: str):
    # xóa các row không cần thiết
    num_of_remove_row = 0
    for i in range(len(sheet)):
        num_of_remove_row += 1
        if isinstance(sheet.iloc[i, 0], str) and 'mathangchuyeu' in clean_text(sheet.iloc[i, 0]): break

    if type == 'import':
        sheet = sheet.iloc[num_of_remove_row:len(sheet) - 1, ::].reset_index(drop =True)
        
    else: sheet = sheet.iloc[num_of_remove_row::, ::].reset_index(drop =True)

    name_colums = ['product_name', 'luong_thang_truoc', 'gia_tri_thang_truoc', 'luong_thang_nay', 'gia_tri_thang_nay']
    # xoa cac cot kh can thiet
    sheet = sheet.iloc[::, [1, 2, 3, 5 ,6]]
    sheet.columns = name_colums
    if type == 'import': sheet.loc[ 29, 'product_name'] = 'Ô tô-nguyên chiếc' 
    return sheet
    


In [3]:
df = extract_ecommerce_data(export_sheet, 'e')

In [4]:
df

,product_name,luong_thang_truoc,gia_tri_thang_truoc,luong_thang_nay,gia_tri_thang_nay
0,Thủy sản,NaN,764.041033,NaN,740
1,Rau quả,NaN,326.00348,NaN,330
2,Hạt điều,32.331,293.172854,30,266.737838
3,Cà phê,156.258,296.590486,130,243.958404
4,Chè,11.678,20.428791,12,20.753153
5,Hạt tiêu,22.04,70.541157,22,67.522301
6,Gạo,537.948,281.034695,450,230.266173
7,Sắn và sản phẩm của sắn,169.611,77.813391,150,72.120953
8,Than đá,144.198,20.298252,250,32.346103
9,Dầu thô,313.503,178.816828,444,252


In [5]:
# visualing section
import_sheet

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,12. Hàng hóa nhập khẩu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Nghìn tấn, triệu USD",NaN,NaN,NaN,NaN
3,NaN,NaN,Thực hiện\ntháng 6\nnăm 2018,NaN,NaN,Ước tính\ntháng 7\nnăm 2018,NaN,NaN,Cộng dồn\n7 tháng\nnăm 2018,NaN,NaN,7 tháng năm\n2018 so với cùng\nkỳ năm 2017 (%),NaN,NaN,NaN,NaN,NaN
4,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,Lượng,Trị giá,NaN,Lượng,Trị giá,NaN,Lượng,Trị giá,NaN,Lượng,Trị giá,NaN,utt1,tht1-utt1,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,TỔNG TRỊ GIÁ,NaN,NaN,19045.893971,NaN,NaN,19800,NaN,NaN,130626.069193,NaN,NaN,110.21283,NaN,14700,4345.893971,3.959415
8,NaN,Khu vực kinh tế trong nước,NaN,8094.434575,NaN,NaN,8200,NaN,NaN,54160.477638,NaN,NaN,112.740026,NaN,6200,1894.434575,1.304173
9,NaN,Khu vực có vốn đầu tư NN,NaN,10952,NaN,NaN,11600,NaN,NaN,76465.591555,NaN,NaN,108.490295,NaN,8500,2452,5.916728


In [6]:
export_sheet

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,11. Hàng hóa xuất khẩu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Nghìn tấn, triệu USD"
3,NaN,NaN,Thực hiện\ntháng 6\nnăm 2018,NaN,NaN,Ước tính\ntháng 7\nnăm 2018,NaN,NaN,Cộng dồn\n7 tháng\nnăm 2018,NaN,NaN,7 tháng năm\n2018 so với cùng\nkỳ năm 2017 (%),NaN
4,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,Lượng,Trị giá,NaN,Lượng,Trị giá,NaN,Lượng,Trị giá,NaN,Lượng,Trị giá
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,TỔNG TRỊ GIÁ,NaN,NaN,19845,NaN,NaN,19500,NaN,NaN,133689,NaN,NaN,115.338194
8,NaN,Khu vực kinh tế trong nước,NaN,6024,NaN,NaN,5848,NaN,NaN,39029,NaN,NaN,118.675077
9,NaN,Khu vực có vốn đầu tư NN,NaN,13821,NaN,NaN,13652,NaN,NaN,94660,NaN,NaN,114.016355
